In [ ]:
#---------------------------------------------------------------------------------------------------------------#
#
#  Code: AA_001_Test_Agent_AI_workflow_for_Stock_prices_02_20260311.ipynb
#
#  Objective: Practice and learn how to run an enhanced Multi-Agent group workflow in Google Colab
#
#             Note-01: It is an example of Reasoning and Acting (ReAct) loop.
#             Note-02: The API key comes from Jingru's OpenAI account.
#             Note-03: This Agentic AI workflow first an Agentic AI to conduct a coder's job, then uses a Critic's agent to check results,
#                      at last, asks Visualizer Agent to show visual result.
#
#            Jingru Chen
#            2026-03-12
#
#--------------------------------------------------------------------------------------------------------------#

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Current time in Eastern Time (handles DST automatically)
start_est = datetime.now(ZoneInfo("America/New_York"))

# Print in different formats
print(start_est.strftime("%Y-%m-%d %H:%M:%S %Z"))   # 2025-03-12 14:35:22 EDT
print(start_est.strftime("%Y-%m-%d %I:%M:%S %p %Z")) # 2025-03-12 02:35:22 PM EDT

2026-03-12 00:46:28 EDT
2026-03-12 12:46:28 AM EDT


In [ ]:
!pip install -U "autogen[gemini]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.8 MB/s eta 0:00:00


In [ ]:
!pip install pyautogen[gemini] yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 3.5 MB/s eta 0:00:00


In [ ]:
import os
import datetime
from google.colab import userdata
import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

# 1. Setup
api_key = userdata.get('OPENAI_API_KEY')
config_list = [{"model": "gpt-4o-mini", "api_key": api_key}]
llm_config = {"config_list": config_list}

# Get current year for the YTD calculation
current_year = datetime.date.today().year

# 2. Define the Specialized Agents
# coder = AssistantAgent(
#     name="Coder",
#     llm_config=llm_config,
#     system_message=f"Python Expert. Write yfinance code to fetch data from Jan 1, {current_year} to today."
# )

coder = AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message=f"""Python Expert. When using yfinance for multiple tickers:
    1. Always use yf.download(tickers, start=..., end=..., auto_adjust=True).
    2. IMMEDIATELY flatten the MultiIndex columns using:
       data.columns = data.columns.get_level_values(1) if you want ticker names.
    3. Use 'Close' instead of 'Adj Close' because auto_adjust=True handles it.
    4. Reply TERMINATE when the task is done."""
)

critic = AssistantAgent(
    name="Critic",
    llm_config=llm_config,
    system_message="Financial Analyst. Verify that the return calculation accounts for the exact start of the year. Ensure numbers are accurate."
)

visualizer = AssistantAgent(
    name="Visualizer",
    llm_config=llm_config,
    system_message="Data Scientist. Use matplotlib or seaborn to create a line chart comparing the cumulative returns of the stocks. Save the plot as 'performance.png'."
)

# user_proxy = UserProxyAgent(
#     name="User_Proxy",
#     human_input_mode="NEVER",
#     code_execution_config={"work_dir": "group_tasks", "use_docker": False},
#     is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
# )

user_proxy.initiate_chat(
    manager,
    message=f"Compare TSLA and NVDA YTD performance for 2026. Use yf.download and flatten the columns so you can access 'TSLA' and 'NVDA' directly from the 'Close' price data."
)

# 3. Infrastructure
# Add 'visualizer' to the agent list
groupchat = GroupChat(
    agents=[user_proxy, coder, critic, visualizer],
    messages=[],
    max_round=15
)

manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# 4. Run the task
user_proxy.initiate_chat(
    manager,
    message=f"Compare TSLA and NVDA YTD performance for {current_year}. Provide the percentage return and a visualization chart."
)

User_Proxy (to chat_manager):

Compare TSLA and NVDA YTD performance for 2026. Use yf.download and flatten the columns so you can access 'TSLA' and 'NVDA' directly from the 'Close' price data.

--------------------------------------------------------------------------------

Next speaker: Coder

Coder (to chat_manager):

To fetch data for TSLA and NVDA from January 1, 2026, to today and compare their year-to-date (YTD) performance, you can use the `yfinance` library. The code below demonstrates how to download the data, flatten the columns, and compare the YTD performance for both stocks.

First, make sure you have the `yfinance` library installed. If you haven't installed it yet, you can do so using pip:

```bash
pip install yfinance
```

Now, here’s the code:

```python
import yfinance as yf
import pandas as pd
from datetime import datetime

# Define the tickers and the start date
tickers = ['TSLA', 'NVDA']
start_date = '2026-01-01'
end_date = datetime.now().date().strftime('%Y-%m-%d

ChatResult(chat_id=187616518112959811573115223778810795661, chat_history=[{'content': 'Compare TSLA and NVDA YTD performance for 2026. Provide the percentage return and a visualization chart.', 'role': 'assistant', 'name': 'User_Proxy'}, {'content': '```python\nimport yfinance as yf\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\n# Define tickers and date range\ntickers = ["TSLA", "NVDA"]\nstart_date = "2026-01-01"\nend_date = "2026-12-31"\n\n# Download data\ndata = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)\n\n# Flatten MultiIndex columns\ndata.columns = data.columns.get_level_values(1)\n\n# Calculate YTD performance\nytd_return = ((data[\'Close\'].iloc[-1] - data[\'Close\'].iloc[0]) / data[\'Close\'].iloc[0]) * 100\n\n# Plotting the closing prices\nplt.figure(figsize=(12, 6))\nplt.plot(data[\'Close\'], label=tickers)\nplt.title(\'YTD Performance of TSLA and NVDA in 2026\')\nplt.xlabel(\'Date\')\nplt.ylabel(\'Closing Price\')\nplt.legend(tickers)\n

In [ ]:
from datetime import datetime
end_est = datetime.now(ZoneInfo("America/New_York"))
duration = end_est - start_est

print(f"Started:  {start_est}")
print(f"Finished: {end_est}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-12 00:46:28.866652-04:00
Finished: 2026-03-12 00:50:26.327640-04:00

Duration: 0:03:57.460988
Duration: 237.461 seconds
